# Étape 3 — Distillation GAN (Générateur One-Step)
**Projet** : Single-Source Fast Generation (GENAI ING3)

On distille le teacher SinFusion (50 étapes DDIM) en un **générateur one-step** (student GAN) inspiré de Diffusion2GAN.

```
z_T (bruit gaussien)  ──Generator──►  x_0 (image en 1 seule passe forward)
```

**Dataset d'entraînement** : les 2000 paires `(z_T, x_0)` construites au notebook 02.

**Ce que le GAN apprend** : pour chaque bruit `z_T`, reproduire en une passe ce que le teacher DDIM ferait en 50 étapes.

## 1. Setup

On importe les dépendances, on définit les chemins, et on détecte le GPU.

**Nouveauté par rapport aux notebooks précédents** : on importe `lpips` (loss perceptuelle) et `torch.nn.functional` (loss L1, opérations de base).

In [ ]:
import os, sys, random, time, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    import lpips
except ImportError:
    os.system('pip install lpips -q')
    import lpips

PROJECT_ROOT  = os.path.dirname(os.path.abspath(''))
SINFUSION_DIR = os.path.join(PROJECT_ROOT, 'sinfusion')
os.chdir(SINFUSION_DIR)
if SINFUSION_DIR not in sys.path:
    sys.path.insert(0, SINFUSION_DIR)

from models.nextnet import NextNet

# ── À CHANGER pour une nouvelle image ────────────────────────────────────────
IMAGE_NAME = 'fruit.png'
# ─────────────────────────────────────────────────────────────────────────────
RUN_NAME = os.path.splitext(IMAGE_NAME)[0] + '_teacher'
ODE_DIR  = os.path.join(PROJECT_ROOT, 'outputs', IMAGE_NAME, RUN_NAME, 'ode_dataset')
OUT_DIR  = os.path.join(PROJECT_ROOT, 'outputs', IMAGE_NAME, RUN_NAME, 'distillation')
os.makedirs(os.path.join(OUT_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'samples'),     exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Dataset ODE → DataLoader

On charge les 2000 paires `(z_T, x_0)` générées à l'étape 2.

### Normalisation
- `z_T` : float32, range ~ [-4, 4], on le garde tel quel (le générateur apprend à gérer cette entrée)
- `x_0` : uint8 [0, 255] → float32 [-1, 1] via `pixel / 127.5 - 1`  
  Pourquoi [-1, 1] ? C'est la convention de SinFusion et des GANs en général. La fonction `tanh` en sortie du générateur produce naturellement cette plage.

### Augmentation : flip horizontal
On applique le même flip simultanément sur `z_T` et `x_0`. Cela **double la variété effective** du dataset (2000 → 4000 paires effectives) sans briser la cohérence : le bruit flippé donne bien l'image flippée.

> On ne fait **pas** de crop ou de rotation car les paires ODE sont des images complètes — un crop casserait la correspondance bruit/image.

In [ ]:
class ODEPairDataset(Dataset):
    """
    Dataset PyTorch pour les paires (z_T, x_0) du dataset ODE.
    z_T : (3, H, W) float32  — bruit gaussien
    x_0 : (3, H, W) float32  — image générée par le teacher, normalisée [-1, 1]
    """
    def __init__(self, ode_dir, augment=True):
        self.noise_files = sorted(glob.glob(os.path.join(ode_dir, 'noise',  '*.npy')))
        self.image_files = sorted(glob.glob(os.path.join(ode_dir, 'images', '*.png')))
        assert len(self.noise_files) == len(self.image_files), 'Mismatch paires ODE !'
        self.augment = augment

    def __len__(self):
        return len(self.noise_files)

    def __getitem__(self, idx):
        z_T = torch.from_numpy(np.load(self.noise_files[idx]))               # (3, H, W) float32
        img = np.array(Image.open(self.image_files[idx]))                     # (H, W, 3) uint8
        x_0 = torch.from_numpy(img).permute(2, 0, 1).float() / 127.5 - 1.0  # (3, H, W) [-1, 1]

        if self.augment and random.random() > 0.5:
            z_T = torch.flip(z_T, dims=[2])  # flip horizontal
            x_0 = torch.flip(x_0, dims=[2])

        return z_T, x_0


dataset = ODEPairDataset(ODE_DIR, augment=True)

# Dimensions de l'image (utile pour instancier fixed_z plus tard)
sample_z, sample_x = dataset[0]
H, W = sample_z.shape[1], sample_z.shape[2]

print(f'Dataset chargé : {len(dataset)} paires')
print(f'Dimensions     : H={H}, W={W}')
print(f'z_T shape      : {sample_z.shape}  range [{sample_z.min():.2f}, {sample_z.max():.2f}]')
print(f'x_0 shape      : {sample_x.shape}  range [{sample_x.min():.2f}, {sample_x.max():.2f}]')

## 3. Générateur (Student)

### Architecture : NextNet sans conditionnement temporel

On **réutilise le backbone NextNet** de SinFusion. Pourquoi ?
- SinFusion l'a entraîné à débruiter des images — exactement le mapping `z_T → x_0` qu'on veut apprendre
- C'est une architecture entièrement convolutionnelle, bien adaptée aux petits datasets single-source
- On évite de redéfinir un réseau from scratch

### Adaptation : `t=0` constant
NextNet prend normalement `(x, t)` avec `t` le timestep de diffusion.  
Ici on passe toujours `t=0`. Le sinusoïdal embedding produit alors un vecteur fixe  
`[sin(0), cos(0), ...] = [0, 1, ...]` qui sera absorbé comme un **biais constant** dans les poids.  
Le réseau apprend à le traiter comme une condition fixe, ce qui est équivalent à ne pas avoir de conditionnement temporel du tout.

**Paramètres** : `depth=16`, `filters=64` — identiques au teacher (~6M paramètres).

In [ ]:
class Generator(nn.Module):
    """
    Générateur one-step : z_T → x_0 en une seule passe forward.
    Wraps NextNet de SinFusion avec t=0 constant.

    Entrée  : z_T  (B, 3, H, W)  bruit gaussien
    Sortie  : x_0  (B, 3, H, W)  image générée, range [-1, 1]
    """
    def __init__(self, filters=64, depth=16):
        super().__init__()
        self.net = NextNet(in_channels=3, filters_per_layer=filters, depth=depth)

    def forward(self, z_T):
        t = torch.zeros(z_T.shape[0], dtype=torch.long, device=z_T.device)  # t=0 constant
        return self.net(z_T, t).clamp(-1., 1.)


G = Generator(filters=64, depth=16).to(device)
n_params_G = sum(p.numel() for p in G.parameters())
print(f'Générateur — paramètres : {n_params_G / 1e6:.2f} M')

# Test forward
with torch.no_grad():
    z_test = torch.randn(2, 3, H, W, device=device)
    x_test = G(z_test)
print(f'Test forward — entrée {z_test.shape} → sortie {x_test.shape}  range [{x_test.min():.2f}, {x_test.max():.2f}]')

## 4. Discriminateur (PatchGAN conditionnel)

### Pourquoi un PatchGAN ?
Un discriminateur PatchGAN produit une **grille de scores locaux** plutôt qu'un scalaire global.  
Chaque score juge un patch de l'image. Cela :
- Force le générateur à produire des textures réalistes **partout** dans l'image (pas juste globalement)
- Est plus stable sur petits datasets qu'un discriminateur global
- Nécessite moins de paramètres (architecture légère)

### Pourquoi conditionnel sur z_T ?
Si le discriminateur ne voit que `x_0`, le générateur peut tricher en produisant de belles images  
**sans lien avec le bruit d'entrée** (mode collapse : toujours la même image, très réaliste).  
En lui donnant aussi `z_T`, il juge la **cohérence de la paire** `(bruit → image)`.  
Concrètement : `input_D = concat(z_T, x)` soit **6 canaux** en entrée.

### Normalisation spectrale
On applique `spectral_norm` sur chaque couche conv. Cela contrôle la constante de Lipschitz du discriminateur,  
ce qui stabilise l'entraînement GAN [Miyato et al., 2018].

In [ ]:
class PatchDiscriminator(nn.Module):
    """
    Discriminateur PatchGAN conditionnel avec normalisation spectrale.

    Entrée  : concat(z_T, x)  (B, 6, H,   W  )
    Sortie  : patch logits    (B, 1, H/8, W/8 )
    """
    def __init__(self, in_channels=6, ndf=64):
        super().__init__()

        def conv_block(in_ch, out_ch, stride=2):
            return nn.Sequential(
                nn.utils.spectral_norm(
                    nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=stride, padding=1, bias=False)
                ),
                nn.LeakyReLU(0.2, inplace=True),
            )

        self.model = nn.Sequential(
            conv_block(in_channels, ndf,     stride=2),  # (B, 64,  H/2,  W/2)
            conv_block(ndf,         ndf * 2, stride=2),  # (B, 128, H/4,  W/4)
            conv_block(ndf * 2,     ndf * 4, stride=2),  # (B, 256, H/8,  W/8)
            conv_block(ndf * 4,     ndf * 8, stride=1),  # (B, 512, H/8,  W/8)  stride=1 : pas de downsampling
            nn.utils.spectral_norm(
                nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1)  # (B, 1, H/8, W/8)
            ),
        )

    def forward(self, z_T, x):
        return self.model(torch.cat([z_T, x], dim=1))


D = PatchDiscriminator(in_channels=6, ndf=64).to(device)
n_params_D = sum(p.numel() for p in D.parameters())
print(f'Discriminateur — paramètres : {n_params_D / 1e6:.2f} M')

# Test forward
with torch.no_grad():
    z_test = torch.randn(2, 3, H, W, device=device)
    x_test = torch.zeros(2, 3, H, W, device=device)
    logits = D(z_test, x_test)
print(f'Test forward — logits shape : {logits.shape}  (patch map {logits.shape[2]}×{logits.shape[3]})')

## 5. Fonctions de perte

On combine trois losses complémentaires :

| Loss | Formule | Ce qu'elle garantit |
|------|---------|---------------------|
| `L_adv` (Hinge) | `−E[D(z, G(z))]` | Réalisme visuel global (tromper D) |
| `L_LPIPS` (perceptuel) | `LPIPS(G(z), x_0)` | Fidélité aux structures hautes fréquences |
| `L_L1` (pixel) | `‖G(z) − x_0‖₁` | Ancrage bas-fréquence, stabilise l'entraînement |

**Loss totale générateur** : `L_G = W_adv·L_adv + W_lpips·L_LPIPS + W_l1·L_L1`

### Hinge loss
Plus robuste que la Binary Cross-Entropy pour les GANs : pas de gradient saturé, la marge permet une séparation plus nette entre réel et faux.

### LPIPS (VGG)
Mesure la distance dans l'espace de features VGG-16, bien plus corrélée à la perception humaine que MSE ou L1 pixel.  
On le freeze (`requires_grad=False`) — il sert uniquement de fonction de mesure.

### Rôle de L1
Sans L1, le GAN peut produire des images réalistes mais très différentes des x_0 cibles (hallucinations). L1 ancre la sortie du générateur aux x_0 du dataset ODE, ce qui est essentiel pour la fidélité à la source.

In [ ]:
# Perceptual loss LPIPS (VGG-16 features, frozen)
lpips_fn = lpips.LPIPS(net='vgg').to(device)
for p in lpips_fn.parameters():
    p.requires_grad = False
print('LPIPS (VGG) chargé et freezé')


def hinge_D(real_logits, fake_logits):
    """Hinge loss pour le discriminateur : maximise la marge réel/faux."""
    loss_real = torch.relu(1.0 - real_logits).mean()
    loss_fake = torch.relu(1.0 + fake_logits).mean()
    return (loss_real + loss_fake) * 0.5


def hinge_G(fake_logits):
    """Hinge loss pour le générateur : tromper le discriminateur."""
    return -fake_logits.mean()


def compute_G_losses(z_T, x_real, x_fake):
    """Calcule les 3 composantes de la loss du générateur."""
    fake_logits = D(z_T, x_fake)
    l_adv   = hinge_G(fake_logits)
    l_lpips = lpips_fn(x_fake, x_real).mean()
    l_l1    = F.l1_loss(x_fake, x_real)
    return l_adv, l_lpips, l_l1

## 6. Hyperparamètres & Initialisation

### TTUR (Two Time-scale Update Rule)
On utilise `lr_D = 4 × lr_G`. L'intuition : si D apprend trop lentement, G le trompe facilement  
et reçoit des gradients peu informatifs → il stagne. En donnant plus de vitesse à D,  
on maintient une **pression adversariale constante** sur G. [Heusel et al., 2017]

### Betas Adam = (0.0, 0.99)
Standard pour l'entraînement GAN. `beta1=0` supprime le momentum sur le gradient — recommandé  
pour éviter les oscillations dans les jeux min-max.

### `fixed_z` pour visualiser la progression
On génère **une seule fois** 6 bruits aléatoires fixes au début de l'entraînement.  
À chaque log, on regénère les images correspondantes avec ces **mêmes** z_T.  
Cela permet de voir directement l'amélioration du générateur sur des exemples cohérents.

In [ ]:
# ── Hyperparamètres ───────────────────────────────────────────────────────────
N_EPOCHS   = 100     # 100 epochs pour une meilleure convergence
BATCH_SIZE = 16
LR_G       = 1e-4
LR_D       = 4e-4    # TTUR : D apprend 4× plus vite que G
W_ADV      = 1.0
W_LPIPS    = 1.0
W_L1       = 10.0
LOG_EVERY  = 500
SAVE_EVERY = 10
N_VIZ      = 6

# ── DataLoader ────────────────────────────────────────────────────────────────
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)

# ── Optimiseurs ───────────────────────────────────────────────────────────────
opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.0, 0.99))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.0, 0.99))

fixed_z = torch.randn(N_VIZ, 3, H, W, device=device)

steps_per_epoch = len(dataloader)
print(f'Epochs visées    : {N_EPOCHS}')
print(f'Batch size       : {BATCH_SIZE}')
print(f'Steps/epoch      : {steps_per_epoch}')
print(f'lr_G / lr_D      : {LR_G} / {LR_D}')
print(f'LOG_EVERY        : {LOG_EVERY} steps')

## 6b. Reprise depuis un checkpoint\n\n`RESUME = True` → recharge automatiquement le checkpoint le plus récent (G, D, optimiseurs) et reprend à l'epoch suivante.\n\n`RESUME = False` → repart de zéro.\n\nLes `GradScaler` sont aussi initialisés ici — ils gèrent la précision mixte (AMP) pour éviter les underflows en float16.

In [ ]:
RESUME = True   # False pour repartir de zéro

history     = {'loss_D': [], 'loss_G': [], 'l_adv': [], 'l_lpips': [], 'l_l1': []}
START_EPOCH = 1
step        = 0

# GradScalers pour AMP (mixed precision float16/float32)
scaler_G = GradScaler()
scaler_D = GradScaler()

if RESUME:
    ckpts = sorted(glob.glob(os.path.join(OUT_DIR, 'checkpoints', 'student_ep*.pt')))
    if ckpts:
        latest = ckpts[-1]
        ckpt   = torch.load(latest, map_location=device)
        G.load_state_dict(ckpt['G_state'])
        D.load_state_dict(ckpt['D_state'])
        opt_G.load_state_dict(ckpt['opt_G'])
        opt_D.load_state_dict(ckpt['opt_D'])
        START_EPOCH = ckpt['epoch'] + 1
        step        = ckpt['step']
        print(f'Checkpoint chargé  : {os.path.basename(latest)}')
        print(f'Reprise à epoch    : {START_EPOCH}  (step {step})')
        print(f'Epochs restantes   : {N_EPOCHS - ckpt["epoch"]}')
    else:
        print('Aucun checkpoint trouvé → départ from scratch')
else:
    print('RESUME=False → entraînement from scratch (epoch 1)')

## 7. Boucle d'entraînement

### Déroulement d'une itération

```
Pour chaque batch (z_T, x_real) du dataset ODE :

── Étape 1 : Entraîner D ──────────────────────────────────────────
  x_fake        = G(z_T)              [G en mode eval, no_grad]
  real_logits   = D(z_T, x_real)
  fake_logits   = D(z_T, x_fake)      [x_fake détaché : pas de gradient vers G]
  loss_D        = hinge_D(real, fake)
  loss_D.backward() → opt_D.step()

── Étape 2 : Entraîner G ──────────────────────────────────────────
  x_fake        = G(z_T)              [G en mode train, avec gradients]
  l_adv         = hinge_G(D(z_T, x_fake))
  l_lpips       = LPIPS(x_fake, x_real)
  l_l1          = L1(x_fake, x_real)
  loss_G        = W_adv*l_adv + W_lpips*l_lpips + W_l1*l_l1
  loss_G.backward() → opt_G.step()
```

### Ce que signifient les valeurs de loss_D
- `loss_D ≈ 0.5` : D incertain entre réel et faux → **équilibre idéal**
- `loss_D → 0`   : D écrase G (trop fort) → G reçoit des gradients nuls → **risque d'effondrement**
- `loss_D → 1`   : G trompe complètement D → G a peut-être convergé ou D est trop faible

### `G.eval()` vs `G.train()`
On passe G en `.eval()` pendant la mise à jour de D : cela empêche les couches de normalisation  
de mettre à jour leurs statistiques de batch pendant une passe qui ne sert qu'à D.  
On repasse en `.train()` pour la mise à jour de G.

In [ ]:
def show_grid(G, fixed_z, step, epoch, out_dir, save=True):
    """Affiche une grille de générations sur les bruits fixes."""
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z)  # (N_VIZ, 3, H, W) in [-1, 1]
    G.train()

    imgs_np = ((imgs.cpu().clamp(-1, 1) + 1) / 2 * 255).byte().permute(0, 2, 3, 1).numpy()

    fig, axes = plt.subplots(1, N_VIZ, figsize=(3 * N_VIZ, 3.5))
    fig.suptitle(f'Epoch {epoch:03d}  |  Step {step:05d}', fontsize=12)
    for col in range(N_VIZ):
        axes[col].imshow(imgs_np[col])
        axes[col].set_title(f'z #{col}', fontsize=8)
        axes[col].axis('off')
    plt.tight_layout()
    if save:
        plt.savefig(os.path.join(out_dir, 'samples', f'step_{step:05d}.png'),
                    dpi=80, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
import torch
print(f"VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")

In [ ]:
t0 = time.time()

print(f'=== Entraînement : epochs {START_EPOCH} → {N_EPOCHS} ===')
if step == 0:
    show_grid(G, fixed_z, step=0, epoch=0, out_dir=OUT_DIR, save=True)

for epoch in range(START_EPOCH, N_EPOCHS + 1):
    epoch_D, epoch_G = [], []
    epoch_adv, epoch_lpips, epoch_l1 = [], [], []

    for z_T, x_real in dataloader:
        z_T    = z_T.to(device)
        x_real = x_real.to(device)

        # ── Étape 1 : Entraîner le discriminateur (avec AMP) ─────────────────
        G.eval()
        with torch.no_grad():
            with autocast():
                x_fake_d = G(z_T)

        with autocast():
            real_logits = D(z_T, x_real)
            fake_logits = D(z_T, x_fake_d)
            loss_D = hinge_D(real_logits, fake_logits)

        opt_D.zero_grad()
        scaler_D.scale(loss_D).backward()
        scaler_D.step(opt_D)
        scaler_D.update()

        # ── Étape 2 : Entraîner le générateur (avec AMP) ─────────────────────
        G.train()
        with autocast():
            x_fake_g = G(z_T)
            fake_logits_g = D(z_T, x_fake_g)
            l_adv   = hinge_G(fake_logits_g)
            l_lpips = lpips_fn(x_fake_g, x_real).mean()
            l_l1    = F.l1_loss(x_fake_g, x_real)
            loss_G  = W_ADV * l_adv + W_LPIPS * l_lpips + W_L1 * l_l1

        opt_G.zero_grad()
        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

        # ── Logs ─────────────────────────────────────────────────────────────
        step += 1
        epoch_D.append(loss_D.item())
        epoch_G.append(loss_G.item())
        epoch_adv.append(l_adv.item())
        epoch_lpips.append(l_lpips.item())
        epoch_l1.append(l_l1.item())

        if step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            print(f'[ep {epoch:03d} | step {step:05d}]  '
                  f'D={loss_D.item():.3f}  G={loss_G.item():.3f}  '
                  f'adv={l_adv.item():.3f}  lpips={l_lpips.item():.3f}  l1={l_l1.item():.4f}  '
                  f'({elapsed/60:.1f} min)')
            show_grid(G, fixed_z, step, epoch, OUT_DIR, save=True)

    # ── Fin d'epoch ───────────────────────────────────────────────────────────
    history['loss_D'].append(np.mean(epoch_D))
    history['loss_G'].append(np.mean(epoch_G))
    history['l_adv'].append(np.mean(epoch_adv))
    history['l_lpips'].append(np.mean(epoch_lpips))
    history['l_l1'].append(np.mean(epoch_l1))

    if epoch % SAVE_EVERY == 0:
        ckpt_path = os.path.join(OUT_DIR, 'checkpoints', f'student_ep{epoch:03d}.pt')
        torch.save({
            'epoch'  : epoch,
            'step'   : step,
            'G_state': G.state_dict(),
            'D_state': D.state_dict(),
            'opt_G'  : opt_G.state_dict(),
            'opt_D'  : opt_D.state_dict(),
        }, ckpt_path)
        print(f'  → Checkpoint ep{epoch:03d} sauvegardé')

total_time = time.time() - t0
print(f'\nEntraînement terminé en {total_time/60:.1f} min ({total_time/3600:.2f} h)')

## 8. Courbes d'entraînement & Visualisation finale

### Comment lire les courbes

| Courbe | Signe positif | Signe négatif |
|--------|--------------|---------------|
| `loss_D` stable ≈ 0.3–0.6 | Équilibre GAN sain | → 0 : D trop fort ; → 1 : G domine |
| `loss_G` décroissant | G s'améliore | Plateau ou remontée = instabilité |
| `l_lpips` décroissant | Fidélité perceptuelle croissante | Stagnation = générations floues |
| `l_l1` décroissant | Pixel-fidélité croissante | — |

### Grille finale : teacher vs student
On affiche les générations du **student** (1 étape) sur les `fixed_z` pour les comparer  
visuellement avec ce que produisait le teacher (50 étapes).

In [ ]:
epochs_range = list(range(1, len(history['loss_D']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Courbes d'entraînement GAN", fontsize=13)

axes[0].plot(epochs_range, history['loss_D'], color='crimson',   label='loss_D')
axes[0].plot(epochs_range, history['loss_G'], color='steelblue', label='loss_G')
axes[0].set_xlabel('Epoch'); axes[0].set_title('D vs G'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['l_lpips'], color='darkorange', label='LPIPS')
axes[1].set_xlabel('Epoch'); axes[1].set_title('Loss perceptuelle (LPIPS)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_range, history['l_l1'],    color='seagreen',  label='L1')
axes[2].set_xlabel('Epoch'); axes[2].set_title('Loss pixel (L1)'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()

print('\nGénérations finales du student (1 étape) :')
show_grid(G, fixed_z, step=step, epoch=N_EPOCHS, out_dir=OUT_DIR, save=True)

## 9. Sauvegarde du checkpoint final

On sauvegarde un checkpoint complet contenant :
- `G_state` : poids du **générateur** — c'est tout ce dont on a besoin pour l'inférence
- `D_state` : poids du discriminateur — utile uniquement si on reprend l'entraînement
- `opt_G`, `opt_D` : états des optimiseurs (pour reprendre exactement là où on s'est arrêté)
- `config` : hyperparamètres utilisés (pour traçabilité)

> **Pour l'inférence** (notebook 04), seul `G_state` est nécessaire.

In [ ]:
final_ckpt = os.path.join(OUT_DIR, 'checkpoints', 'student_final.pt')
torch.save({
    'epoch'  : N_EPOCHS,
    'step'   : step,
    'G_state': G.state_dict(),
    'D_state': D.state_dict(),
    'opt_G'  : opt_G.state_dict(),
    'opt_D'  : opt_D.state_dict(),
    'config' : {
        'IMAGE_NAME'  : IMAGE_NAME,
        'RUN_NAME'    : RUN_NAME,
        'H'           : H,
        'W'           : W,
        'filters'     : 64,
        'depth'       : 16,
        'N_EPOCHS'    : N_EPOCHS,
        'BATCH_SIZE'  : BATCH_SIZE,
        'LR_G'        : LR_G,
        'LR_D'        : LR_D,
        'W_ADV'       : W_ADV,
        'W_LPIPS'     : W_LPIPS,
        'W_L1'        : W_L1,
    }
}, final_ckpt)

print(f'Checkpoint final sauvegardé : {final_ckpt}')
print(f'\nPour charger le générateur en inférence :')
print(f"  ckpt = torch.load('student_final.pt')")
print(f"  G.load_state_dict(ckpt['G_state'])")
print(f"  G.eval()")
print(f"  with torch.no_grad(): x_0 = G(z_T)")